# 第 2 章:DataLoader + 词嵌入 + 位置编码

**核心问题**:token id 序列如何变成模型真正能计算的连续向量?

**章节内容**
1. 滑动窗口数据集 —— 把文本切成 (input, target) 训练对
2. `nn.Embedding` 是什么 —— 从查表理解嵌入层的本质
3. 位置编码 —— 告诉模型每个 token 在序列中的位置
4. 把两者相加,得到最终输入向量
5. 封装到 `llm/dataset.py`

**前置**:确保 `llm/tokenizer.py` 已存在(第 1 章产出)。

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..').resolve()))

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from llm.tokenizer import Tokenizer

tokenizer = Tokenizer()
print('Tokenizer 加载成功,词表大小:', tokenizer.vocab_size)

Tokenizer 加载成功,词表大小: 50257


---
## 1. 滑动窗口数据集

GPT 是**自回归**语言模型:给前 n 个 token,预测第 n+1 个。

训练数据的构造方式:
```
原始 token 序列: [t0, t1, t2, t3, t4, t5, ...]

context_length=4, stride=1:
  input:  [t0, t1, t2, t3]   target: [t1, t2, t3, t4]
  input:  [t1, t2, t3, t4]   target: [t2, t3, t4, t5]
  input:  [t2, t3, t4, t5]   target: [t3, t4, t5, t6]
  ...
```

target 就是 input 向右平移一位 —— 每个位置都在预测「下一个 token」。

In [2]:
# 先用一小段文本理解滑动窗口
raw_text = 'In the beginning God created the heavens and the earth. '\
           'Now the earth was formless and empty.'

token_ids = tokenizer.encode(raw_text)
print('token 总数:', len(token_ids))
print('前 10 个 id:', token_ids[:10])

context_length = 8

# 手动看第一个窗口
x0 = token_ids[:context_length]
y0 = token_ids[1:context_length + 1]
print('\n第一个窗口:')
print('input :', x0)
print('target:', y0)
print('input 文本 :', tokenizer.decode(x0))
print('target 文本:', tokenizer.decode(y0))

token 总数: 20
前 10 个 id: [818, 262, 3726, 1793, 2727, 262, 27636, 290, 262, 4534]

第一个窗口:
input : [818, 262, 3726, 1793, 2727, 262, 27636, 290]
target: [262, 3726, 1793, 2727, 262, 27636, 290, 262]
input 文本 : In the beginning God created the heavens and
target 文本:  the beginning God created the heavens and the


In [10]:
import torch
from torch.utils.data import Dataset

# TODO: 实现 GPTDataset
#
# 参数:
#   token_ids     : List[int]  —— 整篇文本编码后的 id 序列
#   context_length: int        —— 每个样本的长度
#   stride        : int        —— 相邻窗口的步长(stride=1 数据最多但有重叠)
#
# __getitem__(i) 返回:
#   x = tensor([token_ids[i*stride : i*stride+context_length]])
#   y = tensor([token_ids[i*stride+1 : i*stride+context_length+1]])

class GPTDataset(Dataset):
    def __init__(self, token_ids, context_length, stride):
        self.inputs = []
        self.targets = []
        # TODO: 用循环或列表推导式,把所有滑动窗口存入 self.inputs / self.targets
        # 提示: for i in range(0, len(token_ids) - context_length, stride):
        for i in range(0, len(token_ids) - context_length, stride):
            self.inputs.append(torch.tensor(token_ids[i : i+context_length]))
            self.targets.append(torch.tensor(token_ids[i+1 : i+context_length+1]))

    def __len__(self):
        # TODO
        return len(self.inputs)

    def __getitem__(self, idx):
        # TODO: 返回 (self.inputs[idx], self.targets[idx])
        return self.inputs[idx], self.targets[idx]


# 验证
ds = GPTDataset(token_ids, context_length=8, stride=1)
print('数据集样本数:', len(ds))
x, y = ds[0]
print('x shape:', x.shape)  # (8,)
print('y shape:', y.shape)  # (8,)
print('x:', x.tolist())
print('y:', y.tolist())

数据集样本数: 12
x shape: torch.Size([8])
y shape: torch.Size([8])
x: [818, 262, 3726, 1793, 2727, 262, 27636, 290]
y: [262, 3726, 1793, 2727, 262, 27636, 290, 262]


In [11]:
from torch.utils.data import DataLoader

# TODO: 用 DataLoader 包装,batch_size=4,shuffle=True
loader = DataLoader(ds, batch_size=4, shuffle=True)

# 取一个 batch 看看
xb, yb = next(iter(loader))
print('batch x shape:', xb.shape)  # (4, 8)
print('batch y shape:', yb.shape)  # (4, 8)

batch x shape: torch.Size([4, 8])
batch y shape: torch.Size([4, 8])


---
## 2. `nn.Embedding` 的本质

`nn.Embedding(num_embeddings, embedding_dim)` 本质上就是一个**查找表**:

- 内部存了一个形状为 `(num_embeddings, embedding_dim)` 的权重矩阵
- 输入整数 id `i`,输出第 `i` 行向量
- 等价于 `one_hot(i) @ weight_matrix`,但直接用索引更高效

对 GPT 来说:
- `num_embeddings = vocab_size`(每个 token 都有一行)
- `embedding_dim = d_model`(每个 token 映射到 d_model 维的向量)

In [ ]:
vocab_size  = tokenizer.vocab_size  # 50257
d_model     = 64    # 嵌入维度(正式 GPT-2 用 768,我们先用小一点的)

# TODO: 创建 token embedding 层
token_emb = ...  # nn.Embedding(vocab_size, d_model)

# TODO: 把上面取到的 xb 送入 token_emb,得到嵌入向量
x_emb = ...
print('嵌入后 shape:', x_emb.shape)  # (4, 8, 64) = (batch, seq_len, d_model)

# 验证:查表等价于 one-hot 矩阵乘法
idx = xb[0, 0].item()          # 第一个样本第一个 token 的 id
from_table = token_emb.weight[idx]                    # 直接查表
one_hot    = torch.zeros(vocab_size)
one_hot[idx] = 1.0
from_matmul = one_hot @ token_emb.weight              # one-hot 矩阵乘
print('两种方式结果是否一致:', torch.allclose(from_table, from_matmul))

---
## 3. 位置编码

**为什么需要位置编码?**

Transformer 的注意力机制把序列当成一个「集合」来处理 —— 它天然不知道
token 的顺序。位置编码给每个位置一个额外向量,叠加到 token 嵌入上,
让模型区分「第 1 个词」和「第 5 个词」。

**两种常见方案**
- **可学习位置编码**(GPT-2 用这个):再来一个 `nn.Embedding(context_length, d_model)`,
  输入位置下标 0,1,2,...,让模型自己学每个位置的向量。
- **正弦固定编码**(原始 Transformer 论文):用 sin/cos 函数手动生成,不需要训练。

我们先用**可学习位置编码**,和 GPT-2 保持一致。

In [ ]:
context_length = 8   # 与上面 Dataset 保持一致

# TODO: 创建位置编码层
pos_emb = ...  # nn.Embedding(context_length, d_model)

# TODO: 生成位置下标 [0, 1, 2, ..., seq_len-1]
# 提示: torch.arange(xb.shape[1])
positions = ...
print('positions:', positions)  # tensor([0, 1, 2, 3, 4, 5, 6, 7])

# TODO: 把 positions 送入 pos_emb 得到位置向量
p_emb = ...
print('位置编码 shape:', p_emb.shape)  # (8, 64) = (seq_len, d_model)

---
## 4. 相加得到最终输入向量

```
最终输入 = token_embedding + position_embedding
```

`x_emb` 的 shape 是 `(batch, seq_len, d_model)`,
`p_emb` 的 shape 是 `(seq_len, d_model)`。

PyTorch 的广播机制会自动把 `p_emb` 扩展到 batch 维度,直接相加即可。

In [ ]:
# TODO: 把 x_emb 和 p_emb 相加
x_input = ...
print('最终输入 shape:', x_input.shape)  # (4, 8, 64)

# 这个 x_input 就是后面送进 Transformer 的输入!
print('\n数据流总结:')
print(f'  文本  -> token ids  : (batch={xb.shape[0]}, seq={xb.shape[1]})')
print(f'  ids   -> token emb  : (batch, seq, d_model={d_model})')
print(f'  pos   -> pos emb    : (seq, d_model)')
print(f'  token_emb + pos_emb : {x_input.shape}')

---
## 5. 封装到 `llm/dataset.py`

确认上面所有 cell 都跑通后,运行下面的 cell 把代码写入模块文件。

In [ ]:
import pathlib

code = '''
import torch
from torch.utils.data import Dataset, DataLoader


class GPTDataset(Dataset):
    """滑动窗口文本数据集。"""

    def __init__(self, token_ids, context_length, stride):
        self.inputs  = []
        self.targets = []
        for i in range(0, len(token_ids) - context_length, stride):
            self.inputs.append(torch.tensor(token_ids[i : i + context_length]))
            self.targets.append(torch.tensor(token_ids[i + 1 : i + context_length + 1]))

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return self.inputs[idx], self.targets[idx]


def make_dataloader(token_ids, context_length, stride, batch_size, shuffle=True):
    ds = GPTDataset(token_ids, context_length, stride)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, drop_last=True)
'''.lstrip()

out = pathlib.Path('../llm/dataset.py')
out.write_text(code, encoding='utf-8')
print(f'已写入 {out.resolve()}')

# 验证
import importlib, sys
if 'llm.dataset' in sys.modules:
    importlib.reload(sys.modules['llm.dataset'])
from llm.dataset import make_dataloader
from llm.tokenizer import Tokenizer
_tok = Tokenizer()
_ids = _tok.encode('hello world this is a test sentence for the dataloader')
_loader = make_dataloader(_ids, context_length=4, stride=1, batch_size=2)
_xb, _yb = next(iter(_loader))
assert _xb.shape == (2, 4)
print('llm.dataset 验证通过,batch shape:', _xb.shape)

---
## 章末思考题

1. `stride=1` 和 `stride=context_length` 各有什么优劣?实际训练时一般用哪个?
2. 位置编码和 token 嵌入直接相加,为什么不用拼接(concat)?拼接会带来什么问题?
3. 如果序列长度超过 `context_length`,模型能处理吗?会发生什么?

答完就可以进入第 3 章 —— 注意力机制(最关键的一章)。